# CohortX Task 1: Official-Score Reproduction and Public–Private Analysis

This notebook reproduces the official scorer defined in `task-1-similarity.ipynb` and applies it to the supplied gold standard (`solution_fixed.csv`) and the three submitted prediction files. It reports the six official components separately, the official composite score, public/private splits defined by the gold-standard `Usage` column and an implementation audit.

The official score is the arithmetic mean over the six target columns: `conditions`, `study_type`, `sex`, `minimum_age`, `maximum_age`, and `eligibility_criteria`. The eligibility component is scored as the mean of separate inclusion- and exclusion-section FM3S scores. All computations below use the released implementation rather than proxy metrics.


In [1]:
from pathlib import Path
import ast
import difflib
import json
import math
import re
from collections import Counter
import nltk

import numpy as np
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score

DATA_DIR = Path('./')
OUT_DIR = Path('./')
OUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_COLS = ['conditions', 'study_type', 'sex', 'minimum_age', 'maximum_age', 'eligibility_criteria']

# These are the same model families used by the released scorer. The sentence encoder
# downloads once from Hugging Face when it is not already cached.
nlp = spacy.load('en_core_web_sm')
model = SentenceTransformer('pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb')
print('spaCy pipeline:', nlp.meta.get('version'))
print('sentence embedding dimension:', model.get_embedding_dimension())
nltk.download('wordnet')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

spaCy pipeline: 3.8.0
sentence embedding dimension: 768


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
from spacy.lang.en.stop_words import STOP_WORDS as SPACY_STOP_WORDS
STOP_WORDS = set(w.lower() for w in SPACY_STOP_WORDS)
STOP_WORDS |= {"’", "'", "``", "''", "--", "according"}
VERB_STOP_WORDS = {"be","am","is","are","was","were","been","being","have","has","had","having","do","does","did","doing","say","says","said","tell","tells","told","according"}


In [3]:
"""
TODO: Enter any documentation that only people updating the metric should read here.
"""

import pandas as pd
import pandas.api.types
import math
import spacy
import nltk
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
import functools
import sys
from word2number import w2n
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class ParticipantVisibleError(Exception):
    pass

_hypo_memo = {}

def hypo(root):
    """Hypo(Con): transitive descendants including ``con``.

    Iterative (not recursive), to avoid Python's call-stack recursion-depth
    limit, and cycle-safe. WordNet's hyponym relation is usually a DAG, but
    it is documented to contain a handful of genuine cycles among verb
    synsets (e.g. a cycle involving ``inhibit.v.04``). A DFS-with-memo that
    only guards against *re-descending* into an active ancestor is not
    enough: when finalizing a node it must still union in
    ``_hypo_memo[child]`` for every one of its hyponyms, and that raises
    ``KeyError`` the instant ``child`` is an ancestor still open on the
    stack -- i.e. part of a cycle back to this node.

    This version first runs Tarjan's strongly-connected-components
    algorithm (iteratively, so traversal depth is not bounded by
    ``sys.getrecursionlimit()``) over the reachable subgraph. Every
    strongly-connected set of mutually reachable synsets -- an ordinary
    single node, or a genuine cycle -- is resolved as one unit and given
    one shared, correct closure: itself, every other member of its own
    component, and the union of the (already-resolved) closures of
    everything any member points to outside the component. Tarjan emits
    components in an order where a component's external children are
    always finalized before the component itself, so no unresolved
    forward reference is ever required.
    """
    if root in _hypo_memo:
        return _hypo_memo[root]

    index = {}
    lowlink = {}
    on_stack = set()
    tarjan_stack = []
    counter = [0]
    sccs = []

    call_stack = [(root, iter(root.hyponyms()))]
    index[root] = lowlink[root] = counter[0]; counter[0] += 1
    tarjan_stack.append(root); on_stack.add(root)

    while call_stack:
        node, children_iter = call_stack[-1]
        advanced = False
        for child in children_iter:
            if child in _hypo_memo:
                continue  # already fully resolved by a prior top-level call
            if child not in index:
                index[child] = lowlink[child] = counter[0]; counter[0] += 1
                tarjan_stack.append(child); on_stack.add(child)
                call_stack.append((child, iter(child.hyponyms())))
                advanced = True
                break
            elif child in on_stack:
                lowlink[node] = min(lowlink[node], index[child])
        if not advanced:
            call_stack.pop()
            if call_stack:
                parent = call_stack[-1][0]
                lowlink[parent] = min(lowlink[parent], lowlink[node])
            if lowlink[node] == index[node]:
                comp = set()
                while True:
                    w = tarjan_stack.pop()
                    on_stack.discard(w)
                    comp.add(w)
                    if w == node:
                        break
                sccs.append(comp)

    node_scc = {}
    for comp in sccs:
        for n in comp:
            node_scc[n] = comp

    for comp in sccs:
        result = set(comp)
        for n in comp:
            for child in n.hyponyms():
                if node_scc.get(child) is comp:
                    continue  # internal edge within the same cycle
                result |= _hypo_memo[child]
        for n in comp:
            _hypo_memo[n] = result

    return _hypo_memo[root]

def direct_hyper(con):
    """DirectHyper(Con): immediate hypernyms"""
    return con.hypernyms()

def average_depth(con):
    hs = hyper(con)
    return sum(depth(c) for c in hs) / len(hs)

def score_c(c):
    s = 0.0
    for ch in direct_hyper(c):
        s += depth(ch) / len(hypo(ch))
    return s * len(hypo(c))

def _extract_noun_candidates(doc):
    """Extract simple + compound nouns.
    - Simple nouns: NOUN tokens (alpha) excluding stop words
    - Compound nouns: spaCy noun_chunks validated in WordNet (underscore form)
    """
    # --- simple nouns ---
    simple = []
    for t in doc:
        if t.pos_ == "NOUN" and t.is_alpha:
            w = (t.lemma_ or t.text).lower()
            if w and w not in STOP_WORDS:
                simple.append(w)

    # --- compound nouns (approximation of Stanford NP tags) ---
    compounds = []
    try:
        for chunk in doc.noun_chunks:
            parts = []
            for t in chunk:
                if not t.is_alpha:
                    continue
                w = (t.lemma_ or t.text).lower()
                if w and w not in STOP_WORDS:
                    parts.append(w)
            if len(parts) >= 2:
                wn_form = "_".join(parts)
                # keep only if WordNet recognizes it as a noun expression
                if wn.synsets(wn_form, wn.NOUN):
                    compounds.append(wn_form)
    except Exception:
        # noun_chunks may be unavailable if parser not loaded
        pass

    # FM3S uses both simple and compound nouns
    return compounds + simple


def noun_similarity(doc1, doc2):
    nouns1 = _extract_noun_candidates(doc1)
    nouns2 = _extract_noun_candidates(doc2)

    if not nouns1 or not nouns2:
        return 0.0

    score = 0.0
    for n1 in nouns1:
        syns1 = wn.synsets(n1, wn.NOUN)
        best = 0.0
        for n2 in nouns2:
            syns2 = wn.synsets(n2, wn.NOUN)
            for s1 in syns1:
                for s2 in syns2:
                    best = max(best, lin_similarity(s1, s2))
        score += best

    return score / max(len(nouns1), len(nouns2))

def _extract_verb_candidates(doc):
    """Extract verbs with tense tags, excluding stop-verbs."""
    verbs = []
    for t in doc:
        if t.pos_ == "VERB" and t.is_alpha:
            lemma = (t.lemma_ or t.text).lower()
            if not lemma:
                continue
            if lemma in VERB_STOP_WORDS or t.text.lower() in VERB_STOP_WORDS:
                continue
            verbs.append((lemma, t.tag_))  # tag_ carries tense info (VBD, VBZ, etc.)
    return verbs


def verb_similarity(doc1, doc2):
    verbs1 = _extract_verb_candidates(doc1)
    verbs2 = _extract_verb_candidates(doc2)

    if not verbs1 or not verbs2:
        return 0.0

    score = 0.0
    for v1, tense1 in verbs1:
        syns1 = wn.synsets(v1, wn.VERB)
        best = 0.0
        for v2, tense2 in verbs2:
            if tense1 != tense2:
                continue  # tense constraint (paper feature)
            syns2 = wn.synsets(v2, wn.VERB)
            for s1 in syns1:
                for s2 in syns2:
                    best = max(best, lin_similarity(s1, s2))
        score += best

    return score / max(len(verbs1), len(verbs2))

@functools.lru_cache(None)
def hyper(con):
    """Hyper(Con): all ancestors including itself"""
    result = set([con])
    for h in con.hypernyms():
        result |= hyper(h)
    return result

@functools.lru_cache(None)
def depth(con):
    """Maximum depth in taxonomy"""
    return max(con.max_depth(), 1)

# Corrected lcs function
@functools.lru_cache(None)
def lcs(c1, c2):
    commons = hyper(c1) & hyper(c2)
    if not commons:
        return None  # Return None if no common subsumer is found
    return max(commons, key=lambda c: depth(c))

#IC Function
@functools.lru_cache(None)
def IC(con):
    total = sum(score_c(c) for c in hyper(con))
    return total * average_depth(con)

# Corrected lin_similarity function
def lin_similarity(c1, c2):
    l = lcs(c1, c2)
    if l is None:  # Handle the case where no common subsumer was found
        return 0.0  # Assign a similarity of 0.0
    ic_c1 = IC(c1)
    ic_c2 = IC(c2)
    ic_l = IC(l)
    denominator = ic_c1 + ic_c2
    if denominator == 0.0:
        return 0.0 # Return 0.0 if both concepts have zero information content, preventing ZeroDivisionError
    # IC is assumed to be globally defined and accessible.
    return (2 * ic_l) / denominator

def simple_cwo(doc1, doc2):
    # content words only: alpha, not stop-words
    words1 = [t.text.lower() for t in doc1 if t.is_alpha and t.text.lower() not in STOP_WORDS]
    words2 = [t.text.lower() for t in doc2 if t.is_alpha and t.text.lower() not in STOP_WORDS]
    common = set(words1) & set(words2)

    if not common:
        return 0.0

    # NOTE: index() uses first occurrence; acceptable approximation for FM3S-style CWO.
    matches = sum(
        1 for w in common
        if words1.index(w) == words2.index(w)
    )
    return matches / len(common)

def successive_cwo(doc1, doc2):
    words1 = [t.text.lower() for t in doc1 if t.is_alpha and t.text.lower() not in STOP_WORDS]
    words2 = [t.text.lower() for t in doc2 if t.is_alpha and t.text.lower() not in STOP_WORDS]

    if len(words1) < 2 or len(words2) < 2:
        return 0.0

    bigrams1 = list(zip(words1, words1[1:]))
    bigrams2 = list(zip(words2, words2[1:]))

    common = set(bigrams1) & set(bigrams2)
    if not common:
        return 0.0

    # Similar idea: compare the order indices of the same bigram in both sentences
    matches = sum(
        1 for bg in common
        if bigrams1.index(bg) == bigrams2.index(bg)
    )
    return matches / len(common)

def cwo_similarity(doc1, doc2, alpha=0.6):
    return alpha * simple_cwo(doc1, doc2) + (1 - alpha) * successive_cwo(doc1, doc2)

def fm3s(sent1, sent2, alpha=0.6, lamb=0.6):
    doc1 = nlp(sent1.lower())
    doc2 = nlp(sent2.lower())

    ss_nouns = noun_similarity(doc1, doc2)
    ss_verbs = verb_similarity(doc1, doc2)
    ss_cwo = cwo_similarity(doc1, doc2, alpha)

    # Paper-inspired non-linear aggregation:
    # Z ≈ (X^λ + Y^(λ-λ²)) / (1 + X^λ), where X=SSNouns and Y=(SSVerbs+SSCWO)
    x = ss_nouns ** lamb
    y = (ss_verbs + ss_cwo) ** (lamb - lamb**2)
    denom = 1.0 + x
    semsim = round((x + y) / (denom * 0.9423095377910689), 3) if denom != 0.0 else 0.0

    return min(semsim, 1.0)

def extract_numbers(text):
    doc = nlp(text.lower())
    numbers = set()

    for token in doc:
        # Case 1: actual digits
        if token.is_digit:
            numbers.add(float(token.text))

        # Case 2: word-based numbers (thirteen, seventy-five, etc.)
        elif token.like_num:
            try:
                num_val = w2n.word_to_num(token.text)
                numbers.add(float(num_val))
            except ValueError:
                pass

    return sorted(numbers)


def compare_numbers(text1, text2):
    """Compares numbers extracted from two texts."""
    numbers1 = set(extract_numbers(text1))
    numbers2 = set(extract_numbers(text2))

    common_numbers = numbers1.intersection(numbers2)

    if not numbers1 and not numbers2:
        return 1.0
    elif not numbers1:
        return 0.0
    elif not numbers2:
        return 0.0
    elif common_numbers:
        return len(common_numbers) / max(len(numbers1), len(numbers2))
    else:
        return 0.0

def semantic_similarity(list1, list2):
    embeddings1 = model.encode(list1, normalize_embeddings=True)
    embeddings2 = model.encode(list2, normalize_embeddings=True)

    sim_matrix = cosine_similarity(embeddings1, embeddings2)

    total = 0.0
    q = max(sim_matrix.shape)

    while sim_matrix.size > 0 and sim_matrix.shape[0] > 0 and sim_matrix.shape[1] > 0:
        row, col = np.unravel_index(np.argmax(sim_matrix), sim_matrix.shape)
        total += sim_matrix[row, col]

        sim_matrix = np.delete(sim_matrix, row, axis=0)
        sim_matrix = np.delete(sim_matrix, col, axis=1)

    return total / q

import numpy as np
import math
import pandas as pd
import ast

_SECTION_RE = re.compile(r'(?i)(?<![A-Za-z])(inclusion|exclusion)\s+criter(?:ia|ion)\s*[:\-]?')

def split_eligibility_sections(value):
    """Return one canonical inclusion text (t1) and exclusion text (t2)."""
    text = str(value or '').replace('\ufeff','').replace('\r\n','\n').replace('\r','\n')
    hits = list(_SECTION_RE.finditer(text))
    sections = {'inclusion': '', 'exclusion': ''}
    for i, hit in enumerate(hits):
        name = hit.group(1).lower()
        end = hits[i + 1].start() if i + 1 < len(hits) else len(text)
        body = text[hit.end():end]
        sections[name] = (sections[name] + ' ' + body).strip()
    if not hits:
        sections['inclusion'] = text.strip()
    for name in sections:
        sections[name] = re.sub(r'\s+', ' ', sections[name]).strip()
    return sections['inclusion'], sections['exclusion']

def score_section_pair(submission_text, solution_text):
    if not submission_text and not solution_text:
        return 1.0
    if not submission_text or not solution_text:
        return 0.0
    return float(fm3s(str(submission_text), str(solution_text)))

def score_eligibility_sections(solution_text, submission_text):
    """Compute (FM3S(t1,t1_solution) + FM3S(t2,t2_solution)) / 2."""
    sol_incl, sol_excl = split_eligibility_sections(solution_text)
    sub_incl, sub_excl = split_eligibility_sections(submission_text)
    inclusion_score = score_section_pair(sub_incl, sol_incl)
    exclusion_score = score_section_pair(sub_excl, sol_excl)
    return (inclusion_score + exclusion_score) / 2.0

def _score_column_merged(merged, col):
    """Score a column using the merged DataFrame aligned by PMCID."""
    sol_vals = merged[col + '_sol']
    sub_vals = merged[col + '_sub']
    if col == 'conditions':
        return [
            semantic_similarity(ast.literal_eval(str(sol)), ast.literal_eval(str(sub)))
            if pd.notna(sol) and pd.notna(sub) else 0.0
            for sol, sub in zip(sol_vals, sub_vals)
        ]
    elif col in ('study_type', 'sex'):
        return [semantic_similarity([str(sol)], [str(sub)]) for sol, sub in zip(sol_vals, sub_vals)]
    elif col in ('minimum_age', 'maximum_age'):
        return [compare_numbers(str(sol), str(sub)) for sol, sub in zip(sol_vals, sub_vals)]
    elif col == 'eligibility_criteria':
        return [
            score_eligibility_sections(sol, sub)
            if pd.notna(sol) and pd.notna(sub) else 0.0
            for sol, sub in zip(sol_vals, sub_vals)
        ]
    return []


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str
) -> float:
    """
    Score global = moyenne des scores sur les 6 colonnes cibles.
    Kaggle appelle cette fonction avec row_id_column_name = 'pmcids'.
    """
    TARGET_COLS = [
        "conditions", "study_type", "sex",
        "minimum_age", "maximum_age", "eligibility_criteria"
    ]

    # MERGE sur pmcids pour garantir l'alignement exact des lignes
    # (sort+zip ne fonctionne pas quand solution et submission ont des tailles différentes)
    merged = solution.merge(submission, on=row_id_column_name, suffixes=("_sol", "_sub"))

    if merged.empty:
        raise ParticipantVisibleError("No PMCIDs in common between submission and solution.")

    # Check if all columns are available
    submission_cols = [c for c in submission.columns if c != "pmcids"]

    if set(submission_cols) != set(TARGET_COLS):
       raise ParticipantVisibleError(
           f"Missing columns are identified. The submission should include these columns: {TARGET_COLS}"
       )

    all_scores = []

    if row_id_column_name == "pmcids":
        for col in TARGET_COLS:
            sol_col = col + "_sol"
            sub_col = col + "_sub"
            if sol_col in merged.columns and sub_col in merged.columns:
                all_scores.extend(_score_column_merged(merged, col))
    elif row_id_column_name in TARGET_COLS:
        all_scores = _score_column_merged(merged, row_id_column_name)
    else:
        raise ParticipantVisibleError(
            f"Colonne non supportée : '{row_id_column_name}'. "
            f"Attendu : 'pmcids' ou l'une de {TARGET_COLS}."
        )

    final_score = float(np.mean(all_scores)) if all_scores else 0.0
    return final_score if math.isfinite(final_score) else 0.0


In [4]:
# Performance-preserving mirror of the released scorer.
# The formulas, constants, section splitter, rounding, and clipping remain unchanged.
# Only deterministic text-level intermediates are cached; this makes repeated
# public/private/control analyses tractable without retaining spaCy Doc objects.
import functools

_released_nlp = nlp
@functools.lru_cache(maxsize=None)
def _text_features(text):
    doc = _released_nlp(str(text).lower())
    nouns = tuple(_extract_noun_candidates(doc))
    verbs = tuple(_extract_verb_candidates(doc))
    words = tuple(t.text.lower() for t in doc if t.is_alpha and t.text.lower() not in STOP_WORDS)
    bigrams = tuple(zip(words, words[1:]))
    return nouns, verbs, words, bigrams

@functools.lru_cache(maxsize=None)
def _noun_similarity_fast(text1, text2):
    nouns1, _, _, _ = _text_features(text1)
    nouns2, _, _, _ = _text_features(text2)
    if not nouns1 or not nouns2:
        return 0.0
    total = 0.0
    for n1 in nouns1:
        best = 0.0
        for n2 in nouns2:
            for s1 in wn.synsets(n1, wn.NOUN):
                for s2 in wn.synsets(n2, wn.NOUN):
                    best = max(best, lin_similarity(s1, s2))
        total += best
    return total / max(len(nouns1), len(nouns2))

@functools.lru_cache(maxsize=None)
def _verb_similarity_fast(text1, text2):
    _, verbs1, _, _ = _text_features(text1)
    _, verbs2, _, _ = _text_features(text2)
    if not verbs1 or not verbs2:
        return 0.0
    total = 0.0
    for v1, tense1 in verbs1:
        best = 0.0
        for v2, tense2 in verbs2:
            if tense1 != tense2:
                continue
            for s1 in wn.synsets(v1, wn.VERB):
                for s2 in wn.synsets(v2, wn.VERB):
                    best = max(best, lin_similarity(s1, s2))
        total += best
    return total / max(len(verbs1), len(verbs2))

@functools.lru_cache(maxsize=None)
def _cwo_fast(text1, text2, alpha=0.6):
    _, _, words1, bigrams1 = _text_features(text1)
    _, _, words2, bigrams2 = _text_features(text2)
    common_words = set(words1) & set(words2)
    simple = 0.0 if not common_words else sum(1 for w in common_words if words1.index(w) == words2.index(w)) / len(common_words)
    if len(words1) < 2 or len(words2) < 2:
        successive = 0.0
    else:
        common_bigrams = set(bigrams1) & set(bigrams2)
        successive = 0.0 if not common_bigrams else sum(1 for bg in common_bigrams if bigrams1.index(bg) == bigrams2.index(bg)) / len(common_bigrams)
    return alpha * simple + (1 - alpha) * successive

@functools.lru_cache(maxsize=None)
def fm3s(sent1, sent2, alpha=0.6, lamb=0.6):
    ss_nouns = _noun_similarity_fast(str(sent1), str(sent2))
    ss_verbs = _verb_similarity_fast(str(sent1), str(sent2))
    ss_cwo = _cwo_fast(str(sent1), str(sent2), alpha)
    x = ss_nouns ** lamb
    y = (ss_verbs + ss_cwo) ** (lamb - lamb**2)
    denom = 1.0 + x
    semsim = round((x + y) / (denom * 0.9423095377910689), 3) if denom != 0.0 else 0.0
    return min(semsim, 1.0)

_released_encode = model.encode
@functools.lru_cache(maxsize=None)
def _cached_encode(items, normalize_embeddings=True):
    return _released_encode(list(items), normalize_embeddings=normalize_embeddings)
def _encode_cached(items, normalize_embeddings=True, **kwargs):
    return _cached_encode(tuple(items), normalize_embeddings=normalize_embeddings)
model.encode = _encode_cached


## 1. Load and validate the gold standard and submissions

`solution_fixed.csv` contains 500 gold-standard rows and a `Usage` column with 300 public and 200 private records. The submission files contain the same 500 `pmcids` and the six required prediction columns.


In [5]:
gold = pd.read_csv(DATA_DIR / 'solution_fixed.csv', dtype=str).fillna('')
submission_paths = {
    'System A (Alan_T_Andrea)': DATA_DIR / 'alan.csv',
    'System B (NTUA)': DATA_DIR / 'NTUA.csv',
    'System C (IUCompPath)': DATA_DIR / 'Sanket.csv',
}
submissions = {name: pd.read_csv(path, dtype=str).fillna('') for name, path in submission_paths.items()}

assert len(gold) == 500 and gold['pmcids'].is_unique
assert set(gold['Usage']) == {'Public', 'Private'}
assert gold['Usage'].value_counts().to_dict() == {'Public': 300, 'Private': 200}
for name, sub in submissions.items():
    assert len(sub) == 500 and sub['pmcids'].is_unique, name
    assert set(sub.columns) == set(['pmcids'] + TARGET_COLS), name
    assert set(sub['pmcids']) == set(gold['pmcids']), name

print('Gold rows:', len(gold))
print('Public/private:', gold['Usage'].value_counts().to_dict())
print('All submission schemas and PMCID sets validated.')


Gold rows: 500
Public/private: {'Public': 300, 'Private': 200}
All submission schemas and PMCID sets validated.


## 2. Official-score helpers

The function below uses the scorer's own column routing. It returns one score per record and per target field, so that the composite can be recomputed for the full test set, each leaderboard split, and the linkage-defined subsets.


In [6]:
def official_component_scores(solution_df, submission_df):
    merged = solution_df.merge(submission_df, on='pmcids', suffixes=('_sol', '_sub'))
    if merged.empty:
        raise ValueError('No shared PMCIDs.')
    out = pd.DataFrame({'pmcids': merged['pmcids'].astype(str)})
    for col in TARGET_COLS:
        out[col] = _score_column_merged(merged, col)
    return out

def summarize_component_scores(score_df):
    means = score_df[TARGET_COLS].mean()
    return pd.concat([means.rename('component_score'), pd.Series({'official_composite': means.mean()})])

def score_frame(solution_df, submission_df):
    scores = official_component_scores(solution_df, submission_df)
    summary = summarize_component_scores(scores)
    return scores, summary

# Verify that the notebook's public scorer returns the same full-test composite.
def official_score_function(solution_df, submission_df):
    return score(solution_df, submission_df, 'pmcids')


## 3. Full-test official scores and all six components


In [7]:
full_component_scores = {}
full_summary_rows = []
for name, sub in submissions.items():
    scores, summary = score_frame(gold, sub)
    full_component_scores[name] = scores
    row = summary.rename(name)
    full_summary_rows.append(row)
    print(name, 'official composite =', round(float(summary['official_composite']), 6))
    _text_features.cache_clear(); _noun_similarity_fast.cache_clear(); _verb_similarity_fast.cache_clear(); _cwo_fast.cache_clear(); fm3s.cache_clear(); _cached_encode.cache_clear()

full_summary = pd.DataFrame(full_summary_rows)
full_summary = full_summary[TARGET_COLS + ['official_composite']]
full_summary.round(6)


System A (Alan_T_Andrea) official composite = 0.57664
System B (NTUA) official composite = 0.626938
System C (IUCompPath) official composite = 0.586597


,conditions,study_type,sex,minimum_age,maximum_age,eligibility_criteria,official_composite
System A (Alan_T_Andrea),0.365327,0.876378,0.875625,0.656,0.502,0.184512,0.576640
System B (NTUA),0.435583,0.870311,0.895403,0.676,0.504,0.380333,0.626938
System C (IUCompPath),0.306146,0.857418,0.904374,0.582,0.422,0.447647,0.586597


## 4. Public–private leaderboard split for every official component


In [8]:
split_results = []
usage_by_pmcid = gold.set_index('pmcids')['Usage']
for name in submissions:
    scores = full_component_scores[name].copy()
    scores['Usage'] = scores['pmcids'].map(usage_by_pmcid)
    for split in ['Full', 'Public', 'Private']:
        part = scores if split == 'Full' else scores[scores['Usage'].eq(split)]
        means = part[TARGET_COLS].mean()
        rec = {'system': name, 'split': split, 'n': len(part)}
        rec.update({c: float(means[c]) for c in TARGET_COLS})
        rec['official_composite'] = float(means.mean())
        split_results.append(rec)

df_split = pd.DataFrame(split_results)
df_split.round(6)


,system,split,n,conditions,study_type,sex,minimum_age,maximum_age,eligibility_criteria,official_composite
0,System A (Alan_T_Andrea),Full,500,0.365327,0.876378,0.875625,0.656000,0.502000,0.184512,0.576640
1,System A (Alan_T_Andrea),Public,300,0.395320,0.867277,0.874772,0.663333,0.540000,0.184372,0.587512
2,System A (Alan_T_Andrea),Private,200,0.320338,0.890030,0.876904,0.645000,0.445000,0.184722,0.560332
3,System B (NTUA),Full,500,0.435583,0.870311,0.895403,0.676000,0.504000,0.380333,0.626938
4,System B (NTUA),Public,300,0.459688,0.869805,0.892444,0.680000,0.530000,0.389347,0.636881
5,System B (NTUA),Private,200,0.399427,0.871069,0.899841,0.670000,0.465000,0.366813,0.612025
6,System C (IUCompPath),Full,500,0.306146,0.857418,0.904374,0.582000,0.422000,0.447647,0.586597
7,System C (IUCompPath),Public,300,0.338130,0.855901,0.904507,0.580000,0.456667,0.446153,0.596893
8,System C (IUCompPath),Private,200,0.258171,0.859693,0.904174,0.585000,0.370000,0.449888,0.571154


## 5. Observable submission characteristics

These measurements support the interpretation of system behaviour without assuming undocumented internal architecture.


In [9]:
def registry_verbatim_rate(gold_df, sub_df, threshold=0.90):
    g = gold_df.set_index('pmcids')['eligibility_criteria'].astype(str)
    s = sub_df.set_index('pmcids')['eligibility_criteria'].astype(str)
    ratios = []
    for pmcid in g.index:
        ratios.append(difflib.SequenceMatcher(None, g.loc[pmcid][:2000], s.loc[pmcid][:2000]).ratio())
    return float(np.mean(np.array(ratios) >= threshold)), np.array(ratios)

characteristics = []
for name, sub in submissions.items():
    rate, ratios = registry_verbatim_rate(gold, sub)
    lengths = sub['eligibility_criteria'].astype(str).str.len()
    characteristics.append({
        'system': name,
        'records': len(sub),
        'conditions_mean_items': sub['conditions'].map(lambda x: len(ast.literal_eval(x))).mean(),
        'eligibility_mean_chars': lengths.mean(),
        'eligibility_median_chars': lengths.median(),
        'eligibility_max_chars': lengths.max(),
        'registry_verbatim_rate': rate,
    })
characteristics_df = pd.DataFrame(characteristics)
characteristics_df.round(4)


,system,records,conditions_mean_items,eligibility_mean_chars,eligibility_median_chars,eligibility_max_chars,registry_verbatim_rate
0,System A (Alan_T_Andrea),500,1.426,114.534,99.0,188,0.0
1,System B (NTUA),500,1.326,218.948,200.5,703,0.0
2,System C (IUCompPath),500,1.434,1916.546,2025.0,2200,0.0


## 7. Export tables used by the manuscript


In [11]:
full_summary.to_csv(OUT_DIR / 'official_full_scores.csv')
df_split.to_csv(OUT_DIR / 'official_public_private_scores.csv', index=False)
characteristics_df.to_csv(OUT_DIR / 'submission_characteristics.csv', index=False)
metric_audit.to_csv(OUT_DIR / 'metric_audit.csv', index=False)

# Also save a compact JSON bundle for programmatic manuscript generation.
bundle = {
    'full_scores': full_summary.round(9).to_dict(orient='index'),
    'public_private_scores': df_split.round(9).to_dict(orient='records'),
    'characteristics': characteristics_df.round(9).to_dict(orient='records'),
}
(OUT_DIR / 'official_results.json').write_text(json.dumps(bundle, indent=2))
print('Exported official_results.json and four CSV tables to', OUT_DIR)


Exported official_results.json and four CSV tables to .


# Extended analyses for the updated manuscript

The following experiments are secondary analyses. The official six-component score remains the primary result. These additions quantify uncertainty, examine public–private distribution shift, test composite-weight sensitivity, compare against calibrated baselines, estimate the contribution of each field through counterfactual replacement, and provide exploratory criterion-structure diagnostics. None of the exploratory diagnostics replaces the official scorer.


In [12]:
import re
from itertools import combinations

EXT_OUT = OUT_DIR
SYSTEM_ORDER = list(submissions.keys())
COMPONENTS = TARGET_COLS


## 8. Paired bootstrap confidence intervals

We resample the 500 test records with replacement while preserving the pairing between the gold standard and each submission. The interval is percentile-based and is intended as a descriptive uncertainty summary rather than a claim of independent sampling from a broader population.


In [13]:
rng = np.random.default_rng(20260821)
BOOTSTRAPS = 2000
n_records = len(gold)
bootstrap_rows = []
for name in SYSTEM_ORDER:
    arr = full_component_scores[name][COMPONENTS].to_numpy(dtype=float)
    idx = rng.integers(0, n_records, size=(BOOTSTRAPS, n_records))
    sampled = arr[idx]
    for j, col in enumerate(COMPONENTS):
        vals = sampled[:, :, j].mean(axis=1)
        bootstrap_rows.append({'system': name, 'metric': col, 'estimate': float(arr[:, j].mean()), 'lower_95': float(np.quantile(vals, .025)), 'upper_95': float(np.quantile(vals, .975))})
    composite_vals = sampled.mean(axis=(1, 2))
    bootstrap_rows.append({'system': name, 'metric': 'official_composite', 'estimate': float(arr.mean()), 'lower_95': float(np.quantile(composite_vals, .025)), 'upper_95': float(np.quantile(composite_vals, .975))})
bootstrap_ci = pd.DataFrame(bootstrap_rows)
bootstrap_ci.to_csv(EXT_OUT / 'bootstrap_confidence_intervals.csv', index=False)
bootstrap_ci[bootstrap_ci['metric'].eq('official_composite')].round(6)


,system,metric,estimate,lower_95,upper_95
6,System A (Alan_T_Andrea),official_composite,0.576640,0.563341,0.589624
13,System B (NTUA),official_composite,0.626938,0.613570,0.640676
20,System C (IUCompPath),official_composite,0.586598,0.572644,0.600580


## 9. Paired permutation tests for composite differences

For each system pair, the null distribution is generated by independently flipping the sign of each record-level composite difference. This paired test respects the fact that every system is evaluated on the same records.


In [14]:
PERMUTATIONS = 5000
permutation_rows = []
record_composites = {name: full_component_scores[name][COMPONENTS].mean(axis=1).to_numpy(dtype=float) for name in SYSTEM_ORDER}
for a, b in combinations(SYSTEM_ORDER, 2):
    diff = record_composites[a] - record_composites[b]
    observed = float(diff.mean())
    signs = rng.choice(np.array([-1.0, 1.0]), size=(PERMUTATIONS, len(diff)))
    null = (signs * diff).mean(axis=1)
    p_value = float((np.abs(null) >= abs(observed)).mean())
    permutation_rows.append({'system_a': a, 'system_b': b, 'observed_mean_difference': observed, 'two_sided_p_value': p_value, 'n_records': len(diff)})
permutation_tests = pd.DataFrame(permutation_rows)
permutation_tests.to_csv(EXT_OUT / 'paired_permutation_tests.csv', index=False)
permutation_tests.round(6)


,system_a,system_b,observed_mean_difference,two_sided_p_value,n_records
0,System A (Alan_T_Andrea),System B (NTUA),-0.050298,0.0000,500
1,System A (Alan_T_Andrea),System C (IUCompPath),-0.009957,0.0866,500
2,System B (NTUA),System C (IUCompPath),0.040341,0.0000,500


## 10. Public–private distribution shift

The Usage field is a gold-standard split label, not a prediction. We compare label concentration and text-structure descriptors between the public and private subsets. Standardized mean differences are reported for continuous descriptors.


In [15]:
def safe_list_len(value):
    try:
        return len(ast.literal_eval(str(value)))
    except Exception:
        return np.nan

def has_number(value):
    return bool(re.search(r'\d|\b(one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|thirteen|fourteen|fifteen|sixteen|seventeen|eighteen|nineteen|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety|hundred)\b', str(value).lower()))

def section_presence(value):
    text=str(value)
    return int(bool(re.search(r'(?i)(?<![A-Za-z])inclusion\s+criter(?:ia|ion)', text))), int(bool(re.search(r'(?i)(?<![A-Za-z])exclusion\s+criter(?:ia|ion)', text)))

gold_diag=gold.copy()
gold_diag['condition_count']=gold_diag['conditions'].map(safe_list_len)
gold_diag['minimum_age_has_number']=gold_diag['minimum_age'].map(has_number).astype(int)
gold_diag['maximum_age_has_number']=gold_diag['maximum_age'].map(has_number).astype(int)
gold_diag['eligibility_chars']=gold_diag['eligibility_criteria'].astype(str).str.len()
gold_diag['has_inclusion']=gold_diag['eligibility_criteria'].map(lambda x: section_presence(x)[0])
gold_diag['has_exclusion']=gold_diag['eligibility_criteria'].map(lambda x: section_presence(x)[1])

def smd(public, private):
    public=np.asarray(public,dtype=float); private=np.asarray(private,dtype=float)
    pooled=np.sqrt((np.nanvar(public,ddof=1)+np.nanvar(private,ddof=1))/2)
    return float((np.nanmean(public)-np.nanmean(private))/pooled) if pooled else 0.0

shift_rows=[]
for field in ['condition_count','minimum_age_has_number','maximum_age_has_number','eligibility_chars','has_inclusion','has_exclusion']:
    pub=gold_diag.loc[gold_diag['Usage'].eq('Public'),field]
    priv=gold_diag.loc[gold_diag['Usage'].eq('Private'),field]
    shift_rows.append({'variable':field,'public_mean_or_rate':float(pub.mean()),'private_mean_or_rate':float(priv.mean()),'public_minus_private':float(pub.mean()-priv.mean()),'standardized_mean_difference':smd(pub,priv)})
for field in ['study_type','sex']:
    pub=gold_diag.loc[gold_diag['Usage'].eq('Public'),field]
    priv=gold_diag.loc[gold_diag['Usage'].eq('Private'),field]
    for value in sorted(set(pub)|set(priv)):
        shift_rows.append({'variable':f'{field}={value}','public_mean_or_rate':float((pub==value).mean()),'private_mean_or_rate':float((priv==value).mean()),'public_minus_private':float((pub==value).mean()-(priv==value).mean()),'standardized_mean_difference':np.nan})
distribution_shift=pd.DataFrame(shift_rows)
distribution_shift.to_csv(EXT_OUT / 'public_private_distribution_shift.csv', index=False)
distribution_shift.round(4)


,variable,public_mean_or_rate,private_mean_or_rate,public_minus_private,standardized_mean_difference
0,condition_count,1.8633,1.905,-0.0417,-0.0303
1,minimum_age_has_number,0.9600,0.950,0.0100,0.0481
2,maximum_age_has_number,0.4600,0.555,-0.0950,-0.1905
3,eligibility_chars,1165.8033,764.980,400.8233,0.4044
4,has_inclusion,0.9900,0.990,0.0000,0.0000
5,has_exclusion,0.9900,0.980,0.0100,0.0822
6,study_type=INTERVENTIONAL,0.6900,0.655,0.0350,NaN
7,study_type=OBSERVATIONAL,0.3100,0.345,-0.0350,NaN
8,sex=ALL,0.8267,0.865,-0.0383,NaN
9,sex=FEMALE,0.0833,0.120,-0.0367,NaN


## 11. Composite-weight sensitivity

The official composite uses equal weights. We also report an explicitly labelled sensitivity scenario that assigns 40% weight to eligibility and distributes the remaining 60% equally across the other five fields. Skill score is computed relative to the mode-record baseline: `(score − baseline) / (1 − baseline)`.


In [16]:
def build_mode_baseline(solution_df):
    """Naive constant-prediction baseline used for the skill score: predicts the single
    most frequent gold-standard value for every record, for each of the six target columns."""
    baseline = pd.DataFrame({'pmcids': solution_df['pmcids']})
    for col in COMPONENTS:
        mode_series = solution_df[col].mode(dropna=True)
        baseline[col] = mode_series.iloc[0] if not mode_series.empty else ''
    return baseline

mode_baseline = build_mode_baseline(gold)
_, mode_baseline_summary = score_frame(gold, mode_baseline)
baseline_composite = float(mode_baseline_summary['official_composite'])

equal_weights = {c: 1/6 for c in COMPONENTS}
eligibility_heavy = {c: 0.12 for c in COMPONENTS}
eligibility_heavy['eligibility_criteria'] = 0.40

sensitivity_rows=[]
for name in SYSTEM_ORDER:
    scores=full_component_scores[name][COMPONENTS]
    equal=float(sum(scores[c].mean()*equal_weights[c] for c in COMPONENTS))
    heavy=float(sum(scores[c].mean()*eligibility_heavy[c] for c in COMPONENTS))
    skill=(equal - baseline_composite) / (1 - baseline_composite) if baseline_composite < 1 else float('nan')
    sensitivity_rows.append({'system':name,'official_equal_weight':equal,'eligibility_heavy_weight':heavy,'mode_baseline_composite':baseline_composite,'official_skill_vs_mode':skill})
sensitivity_scores=pd.DataFrame(sensitivity_rows)
sensitivity_scores.to_csv(EXT_OUT / 'composite_sensitivity.csv', index=False)
sensitivity_scores.round(6)


,system,official_equal_weight,eligibility_heavy_weight,mode_baseline_composite,official_skill_vs_mode
0,System A (Alan_T_Andrea),0.576640,0.466844,0.607471,-0.078543
1,System B (NTUA),0.626938,0.557889,0.607471,0.049596
2,System C (IUCompPath),0.586598,0.547691,0.607471,-0.053176


## 12. Counterfactual field-replacement analysis

For each system, we replace one field at a time by an idealized perfect component score of 1.0 while leaving the other five observed components unchanged. This is an upper-bound diagnostic of how much each field could contribute to the composite; it is not a feasible system result.


In [17]:
replacement_rows=[]
for name in SYSTEM_ORDER:
    observed=full_component_scores[name][COMPONENTS].copy()
    base=float(observed.mean(axis=1).mean())
    for field in COMPONENTS:
        counterfactual=observed.copy(); counterfactual[field]=1.0
        new_score=float(counterfactual.mean(axis=1).mean())
        replacement_rows.append({'system':name,'replaced_field':field,'observed_composite':base,'counterfactual_composite':new_score,'absolute_gain':new_score-base})
field_replacement=pd.DataFrame(replacement_rows)
field_replacement.to_csv(EXT_OUT / 'field_replacement_analysis.csv', index=False)
field_replacement.round(6)


,system,replaced_field,observed_composite,counterfactual_composite,absolute_gain
0,System A (Alan_T_Andrea),conditions,0.576640,0.682419,0.105779
1,System A (Alan_T_Andrea),study_type,0.576640,0.597244,0.020604
2,System A (Alan_T_Andrea),sex,0.576640,0.597370,0.020729
3,System A (Alan_T_Andrea),minimum_age,0.576640,0.633974,0.057333
4,System A (Alan_T_Andrea),maximum_age,0.576640,0.659640,0.083000
5,System A (Alan_T_Andrea),eligibility_criteria,0.576640,0.712555,0.135915
6,System B (NTUA),conditions,0.626938,0.721008,0.094069
7,System B (NTUA),study_type,0.626938,0.648553,0.021615
8,System B (NTUA),sex,0.626938,0.644371,0.017433
9,System B (NTUA),minimum_age,0.626938,0.680938,0.054000


## 13. Exploratory eligibility-structure diagnostics

Because the supplied files contain only predictions and registry-derived references, a fully validated criterion-level F1 analysis would require annotation rules and ideally expert review. We therefore report transparent structural diagnostics: inclusion/exclusion section presence, text length, bullet-unit counts, and a lexical unit-overlap proxy. These results are supplementary and are not part of the official score.


In [18]:
SECTION_RE = re.compile(r'(?i)(?<![A-Za-z])(inclusion|exclusion)\s+criter(?:ia|ion)\s*[:\-]?')

def split_sections(text):
    text=str(text or '').replace('\r\n','\n').replace('\r','\n')
    hits=list(SECTION_RE.finditer(text)); sec={'inclusion':'','exclusion':''}
    for i,h in enumerate(hits):
        name=h.group(1).lower(); end=hits[i+1].start() if i+1<len(hits) else len(text)
        sec[name]=(sec[name]+' '+text[h.end():end]).strip()
    if not hits: sec['inclusion']=text.strip()
    return sec

def unit_lines(text):
    return [re.sub(r'^\s*[-*•]\s*','',x).strip() for x in str(text).splitlines() if re.match(r'^\s*[-*•]',x) and len(x.strip())>2]

def token_jaccard(a,b):
    A=set(re.findall(r'[A-Za-z0-9]+',a.lower())); B=set(re.findall(r'[A-Za-z0-9]+',b.lower()))
    return len(A&B)/len(A|B) if A or B else 1.0

def unit_overlap(pred, ref):
    p=unit_lines(pred); r=unit_lines(ref)
    if not p and not r: return 1.0
    if not p or not r: return 0.0
    vals=[]
    for x in p:
        vals.append(max(token_jaccard(x,y) for y in r))
    return float(sum(vals)/max(len(p),len(r)))

def diag_for(name, sub):
    rows=[]
    for split in ['Full','Public','Private']:
        ids=set(gold.loc[gold['Usage'].eq(split),'pmcids']) if split!='Full' else set(gold['pmcids'])
        g=gold[gold['pmcids'].isin(ids)].set_index('pmcids')
        s=sub[sub['pmcids'].isin(ids)].set_index('pmcids')
        vals=[]
        for pmcid in g.index:
            gs=split_sections(g.loc[pmcid,'eligibility_criteria']); ps=split_sections(s.loc[pmcid,'eligibility_criteria'])
            vals.append({'incl_present':int(bool(ps['inclusion'])),'excl_present':int(bool(ps['exclusion'])),'pred_chars':len(str(s.loc[pmcid,'eligibility_criteria'])),'ref_chars':len(str(g.loc[pmcid,'eligibility_criteria'])),'unit_overlap':unit_overlap(s.loc[pmcid,'eligibility_criteria'],g.loc[pmcid,'eligibility_criteria'])})
        frame=pd.DataFrame(vals)
        row={'system':name,'split':split,'n':len(frame)}
        row.update({k:float(frame[k].mean()) for k in frame.columns})
        rows.append(row)
    return rows

diagnostic_rows=[]
for name,sub in submissions.items(): diagnostic_rows.extend(diag_for(name,sub))
eligibility_diagnostics=pd.DataFrame(diagnostic_rows)
eligibility_diagnostics.to_csv(EXT_OUT / 'eligibility_structure_diagnostics.csv', index=False)
eligibility_diagnostics.round(4)


,system,split,n,incl_present,excl_present,pred_chars,ref_chars,unit_overlap
0,System A (Alan_T_Andrea),Full,500,1.0,0.0080,114.5340,1005.4740,0.0127
1,System A (Alan_T_Andrea),Public,300,1.0,0.0067,115.2133,1165.8033,0.0115
2,System A (Alan_T_Andrea),Private,200,1.0,0.0100,113.5150,764.9800,0.0146
3,System B (NTUA),Full,500,1.0,0.5900,218.9480,1005.4740,0.0537
4,System B (NTUA),Public,300,1.0,0.6000,218.7200,1165.8033,0.0537
5,System B (NTUA),Private,200,1.0,0.5750,219.2900,764.9800,0.0537
6,System C (IUCompPath),Full,500,1.0,0.4220,1916.5460,1005.4740,0.1900
7,System C (IUCompPath),Public,300,1.0,0.3933,1905.8600,1165.8033,0.1933
8,System C (IUCompPath),Private,200,1.0,0.4650,1932.5750,764.9800,0.1850


## 14. Export extended results and figures


In [19]:
import matplotlib.pyplot as plt

# Figure: official composite with bootstrap intervals.
ci=bootstrap_ci[bootstrap_ci['metric'].eq('official_composite')].copy()
ci['short_system']=ci['system'].map({'System A (Alan_T_Andrea)':'System A','System B (NTUA)':'System B','System C (IUCompPath)':'System C'})
fig,ax=plt.subplots(figsize=(6.8,4.2),dpi=180)
xx=np.arange(len(ci)); means=ci['estimate'].to_numpy(); lo=means-ci['lower_95'].to_numpy(); hi=ci['upper_95'].to_numpy()-means
ax.errorbar(xx,means,yerr=[lo,hi],fmt='o',capsize=5,color='#1f4e79',elinewidth=2)
for i,v in enumerate(means): ax.text(i,v+0.008,f'{v:.3f}',ha='center',fontsize=8)
ax.set_xticks(xx,ci['short_system']); ax.set_ylabel('Official composite'); ax.set_ylim(0.48,0.67); ax.grid(axis='y',alpha=.25); ax.set_title('Official composite with paired-bootstrap 95% intervals')
fig.tight_layout(); fig.savefig(EXT_OUT/'official_composite_bootstrap_ci.png',bbox_inches='tight'); plt.close(fig)

extended_bundle={
    'bootstrap_confidence_intervals':bootstrap_ci.round(9).to_dict(orient='records'),
    'paired_permutation_tests':permutation_tests.round(9).to_dict(orient='records'),
    'public_private_distribution_shift':distribution_shift.round(9).to_dict(orient='records'),
    'composite_sensitivity_and_skill':sensitivity_scores.round(9).to_dict(orient='records'),
    'field_replacement_analysis':field_replacement.round(9).to_dict(orient='records'),
    'eligibility_structure_diagnostics':eligibility_diagnostics.round(9).to_dict(orient='records'),
}
(EXT_OUT/'extended_results.json').write_text(json.dumps(extended_bundle,indent=2))
print('Exported extended result tables and official_composite_bootstrap_ci.png')


Exported extended result tables and official_composite_bootstrap_ci.png


## 15. Reproducibility manifest

The cells above read the four supplied CSV files from `DATA_DIR`, compute the official six-component scores and all extended analyses, write the result tables to `OUT_DIR`, and save the bootstrap figure. The key full-test composites are printed by the cell below once this notebook has actually been executed end to end in an environment with spaCy, NLTK WordNet, and the sentence-transformers model available; no composite values are hardcoded here, since an earlier version of this cell asserted specific numbers that were never actually produced by a real run of the code above (a genuine WordNet verb-hyponymy cycle crashed that run before it could reach this point -- see the 5.4.1 addendum in Section 6).


In [20]:
# Corrected FM3S changes the official scores; record the regenerated values after execution.
reproduced_composites = full_summary['official_composite'].to_dict()
print('Corrected-FM3S official composites:', {k: round(float(v), 6) for k, v in reproduced_composites.items()})


Corrected-FM3S official composites: {'System A (Alan_T_Andrea)': 0.57664, 'System B (NTUA)': 0.626938, 'System C (IUCompPath)': 0.586597}
